In [16]:
from pathlib import Path
from pypdf import PdfReader
 
def load_txt(p): return Path(p).read_text(encoding="utf-8", errors="ignore")
 
def load_pdf(p):
    reader = PdfReader(p)
    return "\n".join(page.extract_text() or "" for page in reader.pages)
 
raw = []
for f in Path(r"C:\Git projects\sales_marketing_rag\data\raw").glob("*.txt"):
    raw.append({"source": f.stem, "doc_type": "transcript", "text": load_txt(f)})
for f in Path(r"C:\Git projects\sales_marketing_rag\data\raw").glob("*.pdf"):
    raw.append({"source": f.stem, "doc_type": "10k", "text": load_pdf(f)})
 
print(raw)

[{'source': 'Q1-2026_call', 'doc_type': 'transcript', 'text': 'Operator: At this time, I’d like to welcome everyone to the Coca-Cola Company’s First Quarter 2026 Earnings Results Conference Call. Today’s call is being recorded. [Operator Instructions] I would like to remind everyone that the purpose of this conference is to talk with investors, and therefore, questions from the media will not be addressed. Media participants should contact Coca-Cola’s Media Relations department if they have any questions. I would now like to introduce Todd Beige, Vice President and Head of Investor Relations. Mr.  Beiger, you may now begin.\nTodd Beiger: Good morning, and thank you for joining us. I’m here with Henrique Braun, our Chief Executive Officer; and John Murphy, our President and Chief Financial Officer. We’ve posted schedules under financial information, in the Investors section of our company website. These reconcile certain non-GAAP financial measures that may be referred to this morning t

In [18]:
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter
 
enc = tiktoken.get_encoding("cl100k_base")
n_tokens = lambda s: len(enc.encode(s))
 
splitter = RecursiveCharacterTextSplitter(
    chunk_size=700, chunk_overlap=120,
    length_function=n_tokens,
    separators=["\n\n", "\n", ". ", " ", ""],
)
 
docs = []
for r in raw:
    for i, c in enumerate(splitter.split_text(r["text"])):
        docs.append({**r, "chunk_index": i, "content": c, "token_count": n_tokens(c)})
print(f"{len(docs)} chunks, total tokens {sum(d['token_count'] for d in docs):,}")


73 chunks, total tokens 44,187


In [ ]:
from collections import defaultdict
import os, psycopg

conn = psycopg.connect(os.environ["DATABASE_URL"], autocommit=True)

doc_ids = {}
for r in raw:
    row = conn.execute(
        "INSERT INTO documents (source, doc_type) VALUES (%s,%s) RETURNING id",
        (r["source"], r["doc_type"])).fetchone()
    doc_ids[r["source"]] = row[0]
 
with conn.cursor() as cur:
    cur.executemany(
        "INSERT INTO chunks (document_id, chunk_index, content, token_count) VALUES (%s,%s,%s,%s)",
        [(doc_ids[d["source"]], d["chunk_index"], d["content"], d["token_count"]) for d in docs])
    

print("chunks inserted")

chunks inserted


In [24]:
docs_count = conn.execute("SELECT COUNT(*) FROM documents").fetchone()[0]
chunks_count = conn.execute("SELECT COUNT(*) FROM chunks").fetchone()[0]
print(f"documents: {docs_count}\nchunks: {chunks_count}")

documents: 4
chunks: 73
